In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("linkedin_junior_data_analyst_jobs.csv")

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

df = df.drop_duplicates(subset="link").reset_index(drop=True)

## Clean text

In [4]:
text_cols = [
    "title",
    "company",
    "location",
    "posted_text",
    "applicants",
    "jd",
    "link"
]

for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

df["text"] = (
    df[["title", "company", "location", "jd"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Dates

In [5]:
df["posted_date_ist"] = pd.to_datetime(
    df["posted_date_ist"], errors="coerce"
)

df["days_since_posted"] = pd.to_numeric(
    df["days_since_posted"], errors="coerce"
).astype("Int64")

## Experience

In [6]:
experience_pattern = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|–|—|to)?\s*"
    r"(\d+(?:\.\d+)?)?\s*(years?|yrs?|months?)",
    re.IGNORECASE
)

def get_experience(text):
    match = experience_pattern.search(str(text))
    if not match:
        return pd.Series(["", pd.NA, pd.NA])

    low = float(match.group(1))
    high = match.group(2)
    unit = match.group(3).lower()

    if unit.startswith("month"):
        low /= 12
        high = float(high) / 12 if high else low
    else:
        high = float(high) if high else low

    return pd.Series([
        match.group(0).strip(),
        round(low, 2),
        round(high, 2) if high is not None else pd.NA
    ])

df[["experience_text", "experience_min", "experience_max"]] = (
    df["text"].apply(get_experience)
)

## Skills

In [7]:
skills = [
    "python",
    "sql",
    "excel",
    "power bi",
    "tableau",
    "power query",
    "dax",
    "statistics",
    "aws",
    "sap"
]

skill_cols = []

for skill in skills:
    col = skill.replace(" ", "_")
    skill_cols.append(col)

    pattern = rf"(?<![a-z0-9]){re.escape(skill)}(?![a-z0-9])"

    df[col] = df["text"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).map({True: "Yes", False: "No"})

## Skill summary

In [8]:
df["required_skills"] = df.apply(
    lambda row: ", ".join(
        skill for skill, col in zip(skills, skill_cols)
        if row[col] == "Yes"
    ),
    axis=1
)

df["skill_count"] = df[skill_cols].eq("Yes").sum(axis=1)

## Work type

In [9]:
df["remote"] = df["text"].str.contains(
    r"\bremote\b|\bwfh\b|\bwork from home\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

df["hybrid"] = df["text"].str.contains(
    r"\bhybrid\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

df["onsite"] = df["text"].str.contains(
    r"\bon[- ]?site\b|\bwork from office\b|\bwfo\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

## Job level

In [10]:
df["entry_level_signal"] = df["text"].str.contains(
    r"\bfresher\b|\bfreshers\b|\bentry[- ]level\b|\bgraduate\b|\bjunior\b|\bearly[- ]career\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

df["internship"] = df["text"].str.contains(
    r"\bintern(?:ship)?\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

df["degree_mentioned"] = df["text"].str.contains(
    r"\bbachelor(?:'s)?\b|\bb\.com\b|\bb\.sc\b|\bb\.tech\b|"
    r"\bmaster(?:'s)?\b|\bmba\b|\bm\.com\b|\bm\.sc\b|\bm\.tech\b",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

## Applicants

In [11]:
def get_applicants(value):
    text = str(value).lower()

    if "over 200" in text or "200+" in text:
        return 200

    match = re.search(r"\d[\d,]*", text)

    if match:
        return int(match.group().replace(",", ""))

    return pd.NA

df["applicants_count"] = (
    df["applicants"]
    .apply(get_applicants)
    .astype("Int64")
)

df["applicants_over_200"] = df["applicants"].str.contains(
    r"over\s*200|200\+",
    case=False,
    regex=True,
    na=False
).map({True: "Yes", False: "No"})

## Final columns

In [12]:
final_cols = [
    "title",
    "company",
    "location",
    "posted_date_ist",
    "posted_time_ist",
    "days_since_posted",
    "posted_text",
    "applicants",
    "applicants_count",
    "applicants_over_200",
    "jd",
    "link",
    "experience_text",
    "experience_min",
    "experience_max",
    *skill_cols,
    "required_skills",
    "skill_count",
    "remote",
    "hybrid",
    "onsite",
    "entry_level_signal",
    "internship",
    "degree_mentioned"
]

df = df[[col for col in final_cols if col in df.columns]].copy()

## Save

In [13]:
df.to_csv(
    "linkedin_jobs_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

## Check

In [14]:
df.head()

,title,company,location,posted_date_ist,posted_time_ist,days_since_posted,posted_text,applicants,applicants_count,applicants_over_200,...,aws,sap,required_skills,skill_count,remote,hybrid,onsite,entry_level_signal,internship,degree_mentioned
0,"Right Shore, Revenue Planning Analyst",Teradata,"Navi Mumbai, Maharashtra, India",2026-09-26,05:30:00,1,1 day ago,35 applicants,35,No,...,No,No,"excel, power bi, statistics",3,No,Yes,No,No,No,Yes
1,"Index Projections and Analytics, BlackRock Glo...",BlackRock,"Mumbai, Maharashtra, India",2026-09-26,05:30:00,1,18 hours ago,Over 200 applicants,200,Yes,...,No,No,"python, sql",2,Yes,Yes,No,No,No,Yes
2,Climate Research Analyst,ISS STOXX,Mumbai Metropolitan Region,2026-09-26,05:30:00,1,23 hours ago,Over 200 applicants,200,Yes,...,No,No,excel,1,No,No,No,No,No,Yes
3,Index Data Research Analyst,ISS STOXX,Mumbai Metropolitan Region,2026-09-26,05:30:00,1,1 day ago,111 applicants,111,No,...,No,No,"python, sql, dax",3,No,No,No,No,No,No
4,Data Management Executive – Backend,Altraize,Mumbai Metropolitan Region,2026-09-25,05:30:00,2,1 day ago,Over 200 applicants,200,Yes,...,No,No,excel,1,No,No,No,Yes,No,No
